# 01 _ Data Audit and Cleaning

**Project:** Corporacion Favorita Store Sales Forecasting  
**Project type:** Data Science / MLOps Portfolio Project  
**Notebook:** `notebooks/01_data_audit_cleaning.ipynb`

---

## 1.Purpose

This notebook audits, validates, cleans, and prepares the raw datasets used in the sales forecasting project.

The main goal is to create a reliable data foundation before moving into exploratory data analysis, feature engineering, baseline modeling, and machine learning.


# 2.Notebook Objective

The objective of this notebook is to transform the raw input files into validated, clean, and reusable datasets without modifying the original data.

The notebook will:

- Inspect the raw files available in `data/raw/`.
- Validate schemas, data types, date ranges, missing values, duplicates, and logical keys.
- Check consistency between `train.csv`, `test.csv`, `stores.csv`, `oil.csv`, `holidays_events.csv`, `transactions.csv`, and `sample_submission.csv`.
- Identify data quality risks that could affect forecasting performance.
- Apply only minimal and justified cleaning steps.
- Generate clean intermediate datasets in `data/interim/`.
- Generate modeling-ready base tables in `data/processed/`.
- Save data quality reports in `reports/data_quality/`.

The raw data is never overwritten. Every generated dataset must be reproducible from the original files.


# 3.Scope.

## Scope.

The purpose of this notebook is to ensure that all downstream analysis and modeling steps rely on validated, traceable, and reusable datasets.

It includes:

- Raw file inventory.
- Reproducible data loading.
- Schema validation.
- Date range validation.
- Key uniqueness checks.
- Missing value audit.
- Duplicate record audit.
- Numerical consistency checks.
- Categorical consistency checks.
- Train/test consistency checks.
- Referential integrity checks.
- Join risk assessment.
- Minimal cleaning of auxiliary tables.
- Creation of clean intermediate datasets.
- Creation of modeling-ready base tables.
- Export of data quality reports.

In this project, `family` represents the product category level available in the dataset. There is no separate item-level metadata file.


# 4.Imports and Path Setup

This section imports the required libraries and defines the project paths used throughout the notebook.

The notebook follows a reproducible folder structure:

- Raw data is read from `data/raw/`.
- Clean intermediate datasets are saved in `data/interim/`.
- Modeling-ready base tables are saved in `data/processed/`.
- Data quality reports are saved in `reports/data_quality/`.

The raw data is never modified.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)


# Dependency check
dependency_versions = pd.DataFrame(
    [
        {"package": "pandas", "version": pd.__version__},
        {"package": "numpy", "version": np.__version__},
        {"package": "pyarrow", "version": pa.__version__},
    ]
)

display(dependency_versions)


# Project root

def find_project_root(start_path=None, required_dirs=("data", "notebooks", "reports")):
    """Find the project root by walking upward from the current directory."""
    current_path = Path.cwd() if start_path is None else Path(start_path)
    current_path = current_path.resolve()

    for candidate_path in [current_path, *current_path.parents]:
        if all((candidate_path / directory).exists() for directory in required_dirs):
            return candidate_path

    raise FileNotFoundError(
        "Project root could not be detected. "
        f"Expected a parent directory containing: {required_dirs}. "
        f"Current working directory: {current_path}"
    )


PROJECT_ROOT = find_project_root()

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

# Reports directories
REPORTS_DIR = PROJECT_ROOT / "reports"
DATA_QUALITY_DIR = REPORTS_DIR / "data_quality"


print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Interim data directory: {INTERIM_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Data quality reports directory: {DATA_QUALITY_DIR}")


# 5. Directory Validation

This section validates the project directories required for the data audit workflow. 

The raw data directory must already exist because it contains the original source files. Output directories are created if needed. 


In [2]:
required_paths = [
    "PROJECT_ROOT",
    "RAW_DIR",
    "INTERIM_DIR",
    "PROCESSED_DIR",
    "DATA_QUALITY_DIR",
]

missing_variables = [
    path_name for path_name in required_paths if path_name not in globals()
]

if missing_variables:
    raise NameError(
        "Missing required path variables from the previous setup cell: "
        + ", ".join(missing_variables)
    )


required_input_dirs = {
    "raw_data": RAW_DIR,
}

required_output_dirs = {
    "interim_data": INTERIM_DIR,
    "processed_data": PROCESSED_DIR,
    "data_quality_reports": DATA_QUALITY_DIR,
}

directory_checks = []

for directory_name, directory_path in required_input_dirs.items():
    directory_checks.append(
        {
            "directory_name": directory_name,
            "directory_type": "input",
            "path": str(directory_path),
            "exists": directory_path.exists(),
        }
    )

for directory_name, directory_path in required_output_dirs.items():
    directory_path.mkdir(parents=True, exist_ok=True)

    directory_checks.append(
        {
            "directory_name": directory_name,
            "directory_type": "output",
            "path": str(directory_path),
            "exists": directory_path.exists(),
        }
    )

directory_checks_df = pd.DataFrame(directory_checks)

display(directory_checks_df)

missing_input_dirs = directory_checks_df.loc[
    (directory_checks_df["directory_type"] == "input")
    & (~directory_checks_df["exists"]),
    "path",
].tolist()

if missing_input_dirs:
    raise FileNotFoundError(
        "Missing required input directories: " + ", ".join(missing_input_dirs)
    )


,directory_name,directory_type,path,exists
0,raw_data,input,....,True
1,interim_data,output,....,True
2,processed_data,output,....,True
3,data_quality_reports,output,....,True


# 6.Raw Files Inventory

This section scans the `data/raw/` directory and validates whether the expected source files are available.

The inventory is saved as a data quality report for traceability.


In [3]:
required_objects = ["RAW_DIR", "DATA_QUALITY_DIR", "pd"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


expected_raw_files = [
    "train.csv",
    "test.csv",
    "stores.csv",
    "oil.csv",
    "holidays_events.csv",
    "transactions.csv",
    "sample_submission.csv",
]

raw_files = sorted([file for file in RAW_DIR.iterdir() if file.is_file()])

raw_files_inventory = []

for file_path in raw_files:
    raw_files_inventory.append(
        {
            "file_name": file_path.name,
            "file_extension": file_path.suffix,
            "file_size_mb": round(file_path.stat().st_size / (1024**2), 3),
            "modified_at": pd.Timestamp.fromtimestamp(file_path.stat().st_mtime),
            "is_expected_file": file_path.name in expected_raw_files,
        }
    )

raw_files_inventory_df = pd.DataFrame(raw_files_inventory)

expected_files_validation_df = pd.DataFrame(
    [
        {
            "file_name": file_name,
            "exists": (RAW_DIR / file_name).exists(),
            "path": str(RAW_DIR / file_name),
        }
        for file_name in expected_raw_files
    ]
)

display(raw_files_inventory_df)
display(expected_files_validation_df)

DATA_QUALITY_DIR.mkdir(parents=True, exist_ok=True)

raw_files_inventory_df.to_csv(
    DATA_QUALITY_DIR / "raw_files_inventory.csv",
    index=False,
)

expected_files_validation_df.to_csv(
    DATA_QUALITY_DIR / "expected_raw_files_validation.csv",
    index=False,
)

missing_expected_files = expected_files_validation_df.loc[
    ~expected_files_validation_df["exists"],
    "file_name",
].tolist()

if missing_expected_files:
    raise FileNotFoundError(
        "Missing expected raw files: " + ", ".join(missing_expected_files)
    )


,file_name,file_extension,file_size_mb,modified_at,is_expected_file
0,holidays_events.csv,.csv,0.021,2021-11-22 20:13:44,True
1,oil.csv,.csv,0.020,2021-11-22 20:13:44,True
2,sample_submission.csv,.csv,0.326,2021-11-22 20:13:44,True
3,stores.csv,.csv,0.001,2021-11-22 20:13:44,True
4,test.csv,.csv,0.975,2021-11-22 20:13:44,True
5,train.csv,.csv,116.158,2021-11-22 20:13:46,True
6,transactions.csv,.csv,1.481,2021-11-22 20:13:54,True


,file_name,exists,path
0,train.csv,True,....
1,test.csv,True,....
2,stores.csv,True,....
3,oil.csv,True,....
4,holidays_events.csv,True,....
5,transactions.csv,True,....
6,sample_submission.csv,True,....


# 7.Load Raw Datasets

This section loads the raw CSV files into memory.

Dates are parsed during loading when applicable, but no cleaning or transformations are applied at this stage.


In [4]:
required_objects = ["RAW_DIR", "pd"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


raw_file_paths = {
    "train": RAW_DIR / "train.csv",
    "test": RAW_DIR / "test.csv",
    "stores": RAW_DIR / "stores.csv",
    "oil": RAW_DIR / "oil.csv",
    "holidays": RAW_DIR / "holidays_events.csv",
    "transactions": RAW_DIR / "transactions.csv",
    "sample_submission": RAW_DIR / "sample_submission.csv",
}

missing_files = [
    dataset_name
    for dataset_name, file_path in raw_file_paths.items()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing raw files for datasets: " + ", ".join(missing_files)
    )


date_columns = {
    "train": ["date"],
    "test": ["date"],
    "oil": ["date"],
    "holidays": ["date"],
    "transactions": ["date"],
}


raw_datasets = {}

for dataset_name, file_path in raw_file_paths.items():
    parse_dates = date_columns.get(dataset_name, None)

    raw_datasets[dataset_name] = pd.read_csv(
        file_path,
        parse_dates=parse_dates,
        low_memory=False,
    )


train = raw_datasets["train"]
test = raw_datasets["test"]
stores = raw_datasets["stores"]
oil = raw_datasets["oil"]
holidays = raw_datasets["holidays"]
transactions = raw_datasets["transactions"]
sample_submission = raw_datasets["sample_submission"]


load_summary_df = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "rows": dataset.shape[0],
            "columns": dataset.shape[1],
            "memory_mb": round(dataset.memory_usage(deep=True).sum() / (1024**2), 2),
        }
        for dataset_name, dataset in raw_datasets.items()
    ]
)

display(load_summary_df)


,dataset,rows,columns,memory_mb
0,train,3000888,6,168.16
1,test,28512,5,1.38
2,stores,54,5,0.00
3,oil,1218,2,0.02
4,holidays,350,6,0.03
5,transactions,83488,3,1.91
6,sample_submission,28512,2,0.44


# 8. First Technical Overview

This section creates a first technical summary of each raw dataset.

The goal is to understand the structure, size, data types, missing values, duplicates rows, and available date ranges before applaying any cleaning decision.


In [5]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


dataset_overview_rows = []
column_overview_rows = []
date_range_rows = []

for dataset_name, dataset in raw_datasets.items():
    rows = dataset.shape[0]
    columns = dataset.shape[1]
    memory_mb = round(dataset.memory_usage(deep=True).sum() / (1024**2), 2)
    duplicated_rows = int(dataset.duplicated().sum())
    total_missing_values = int(dataset.isna().sum().sum())
    missing_values_pct = (
        round(total_missing_values / (rows * columns) * 100, 4)
        if rows > 0 and columns > 0
        else 0
    )

    dataset_overview_rows.append(
        {
            "dataset": dataset_name,
            "rows": rows,
            "columns": columns,
            "memory_mb": memory_mb,
            "duplicated_rows": duplicated_rows,
            "total_missing_values": total_missing_values,
            "missing_values_pct": missing_values_pct,
        }
    )

    for column_name in dataset.columns:
        missing_values = int(dataset[column_name].isna().sum())
        missing_pct = round(missing_values / rows * 100, 4) if rows > 0 else 0
        unique_values = int(dataset[column_name].nunique(dropna=True))

        column_overview_rows.append(
            {
                "dataset": dataset_name,
                "column_name": column_name,
                "dtype": str(dataset[column_name].dtype),
                "missing_values": missing_values,
                "missing_pct": missing_pct,
                "unique_values": unique_values,
            }
        )

    possible_date_columns = [
        column_name for column_name in dataset.columns if "date" in column_name.lower()
    ]

    for date_column in possible_date_columns:
        parsed_dates = pd.to_datetime(dataset[date_column], errors="coerce")

        date_range_rows.append(
            {
                "dataset": dataset_name,
                "date_column": date_column,
                "min_date": parsed_dates.min(),
                "max_date": parsed_dates.max(),
                "invalid_dates": int(parsed_dates.isna().sum()),
                "unique_dates": int(parsed_dates.nunique(dropna=True)),
            }
        )


dataset_overview_df = pd.DataFrame(dataset_overview_rows)
column_overview_df = pd.DataFrame(column_overview_rows)
date_ranges_df = pd.DataFrame(date_range_rows)


display(dataset_overview_df)
display(column_overview_df)
display(date_ranges_df)


dataset_overview_df.to_csv(
    DATA_QUALITY_DIR / "dataset_overview.csv",
    index=False,
)

column_overview_df.to_csv(
    DATA_QUALITY_DIR / "column_overview.csv",
    index=False,
)

date_ranges_df.to_csv(
    DATA_QUALITY_DIR / "date_ranges_overview.csv",
    index=False,
)


,dataset,rows,columns,memory_mb,duplicated_rows,total_missing_values,missing_values_pct
0,train,3000888,6,168.16,0,0,0.0000
1,test,28512,5,1.38,0,0,0.0000
2,stores,54,5,0.00,0,0,0.0000
3,oil,1218,2,0.02,0,43,1.7652
4,holidays,350,6,0.03,0,0,0.0000
5,transactions,83488,3,1.91,0,0,0.0000
6,sample_submission,28512,2,0.44,0,0,0.0000


,dataset,column_name,dtype,missing_values,missing_pct,unique_values
0,train,id,int64,0,0.0000,3000888
1,train,date,datetime64[us],0,0.0000,1684
2,train,store_nbr,int64,0,0.0000,54
3,train,family,str,0,0.0000,33
4,train,sales,float64,0,0.0000,379610
5,train,onpromotion,int64,0,0.0000,362
6,test,id,int64,0,0.0000,28512
7,test,date,datetime64[us],0,0.0000,16
8,test,store_nbr,int64,0,0.0000,54
9,test,family,str,0,0.0000,33


,dataset,date_column,min_date,max_date,invalid_dates,unique_dates
0,train,date,2013-01-01,2017-08-15,0,1684
1,test,date,2017-08-16,2017-08-31,0,16
2,oil,date,2013-01-01,2017-08-31,0,1218
3,holidays,date,2012-03-02,2017-12-26,0,312
4,transactions,date,2013-01-01,2017-08-15,0,1682


# 9.Expected Schema Validation

This section validates whether each raw dataset contains the expected columns for the project.

The goal is to verify the input data contract before applying cleaning rules or creating downstream datasets.


In [6]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


expected_schemas = {
    "train": [
        "id",
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion",
    ],
    "test": [
        "id",
        "date",
        "store_nbr",
        "family",
        "onpromotion",
    ],
    "stores": [
        "store_nbr",
        "city",
        "state",
        "type",
        "cluster",
    ],
    "oil": [
        "date",
        "dcoilwtico",
    ],
    "holidays": [
        "date",
        "type",
        "locale",
        "locale_name",
        "description",
        "transferred",
    ],
    "transactions": [
        "date",
        "store_nbr",
        "transactions",
    ],
    "sample_submission": [
        "id",
        "sales",
    ],
}


schema_validation_rows = []

for dataset_name, expected_columns in expected_schemas.items():
    df = raw_datasets[dataset_name]
    actual_columns = list(df.columns)

    missing_columns = sorted(set(expected_columns) - set(actual_columns))
    unexpected_columns = sorted(set(actual_columns) - set(expected_columns))

    schema_validation_rows.append(
        {
            "dataset": dataset_name,
            "expected_columns": expected_columns,
            "actual_columns": actual_columns,
            "missing_columns": missing_columns,
            "unexpected_columns": unexpected_columns,
            "schema_is_valid": len(missing_columns) == 0
            and len(unexpected_columns) == 0,
        }
    )


schema_validation_df = pd.DataFrame(schema_validation_rows)

display(schema_validation_df)

schema_validation_df.to_csv(
    DATA_QUALITY_DIR / "schema_validation.csv",
    index=False,
)

invalid_schemas = schema_validation_df.loc[
    ~schema_validation_df["schema_is_valid"],
    "dataset",
].tolist()

if invalid_schemas:
    raise ValueError(
        "Invalid schemas detected in datasets: " + ", ".join(invalid_schemas)
    )


,dataset,expected_columns,actual_columns,missing_columns,unexpected_columns,schema_is_valid
0,train,"[id, date, store_nbr, family, sales, onpromotion]","[id, date, store_nbr, family, sales, onpromotion]",[],[],True
1,test,"[id, date, store_nbr, family, onpromotion]","[id, date, store_nbr, family, onpromotion]",[],[],True
2,stores,"[store_nbr, city, state, type, cluster]","[store_nbr, city, state, type, cluster]",[],[],True
3,oil,"[date, dcoilwtico]","[date, dcoilwtico]",[],[],True
4,holidays,"[date, type, locale, locale_name, description,...","[date, type, locale, locale_name, description,...",[],[],True
5,transactions,"[date, store_nbr, transactions]","[date, store_nbr, transactions]",[],[],True
6,sample_submission,"[id, sales]","[id, sales]",[],[],True


# 10.Date Range Audit

This section validates the temporal coverage of each dataset.

For a forecasting project, date consistency is critical because the model must learn only from historical data and predict future dates without leakage.


In [7]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


datasets_with_dates = {
    "train": "date",
    "test": "date",
    "oil": "date",
    "holidays": "date",
    "transactions": "date",
}

date_range_rows = []

for dataset_name, date_column in datasets_with_dates.items():
    df = raw_datasets[dataset_name].copy()

    parsed_dates = pd.to_datetime(df[date_column], errors="coerce")

    min_date = parsed_dates.min()
    max_date = parsed_dates.max()
    unique_dates = parsed_dates.nunique(dropna=True)
    invalid_dates = parsed_dates.isna().sum()

    expected_calendar_days = (
        (max_date - min_date).days + 1
        if pd.notna(min_date) and pd.notna(max_date)
        else None
    )

    missing_calendar_days = (
        expected_calendar_days - unique_dates
        if expected_calendar_days is not None
        else None
    )

    date_range_rows.append(
        {
            "dataset": dataset_name,
            "date_column": date_column,
            "min_date": min_date,
            "max_date": max_date,
            "unique_dates": unique_dates,
            "expected_calendar_days": expected_calendar_days,
            "missing_calendar_days": missing_calendar_days,
            "invalid_dates": int(invalid_dates),
        }
    )


date_range_audit_df = pd.DataFrame(date_range_rows)

display(date_range_audit_df)

date_range_audit_df.to_csv(
    DATA_QUALITY_DIR / "date_range_audit.csv",
    index=False,
)


train_dates = pd.to_datetime(raw_datasets["train"]["date"])
test_dates = pd.to_datetime(raw_datasets["test"]["date"])

train_min_date = train_dates.min()
train_max_date = train_dates.max()
test_min_date = test_dates.min()
test_max_date = test_dates.max()

train_unique_dates = train_dates.nunique()
test_unique_dates = test_dates.nunique()

temporal_validation = {
    "train_min_date": train_min_date,
    "train_max_date": train_max_date,
    "test_min_date": test_min_date,
    "test_max_date": test_max_date,
    "train_unique_dates": train_unique_dates,
    "test_unique_dates": test_unique_dates,
    "test_horizon_days": test_unique_dates,
    "test_starts_after_train": test_min_date > train_max_date,
    "train_test_overlap_days": len(set(train_dates.dt.date) & set(test_dates.dt.date)),
}

temporal_validation_df = pd.DataFrame([temporal_validation])

display(temporal_validation_df)

temporal_validation_df.to_csv(
    DATA_QUALITY_DIR / "temporal_validation.csv",
    index=False,
)

if temporal_validation["train_test_overlap_days"] > 0:
    raise ValueError("Train and test contain overlapping dates.")

if not temporal_validation["test_starts_after_train"]:
    raise ValueError("Test dates do not start after the training period.")

if temporal_validation["test_horizon_days"] != 16:
    raise ValueError(
        "Unexpected test horizon. "
        f"Expected 16 unique dates, found {temporal_validation['test_horizon_days']}."
    )


,dataset,date_column,min_date,max_date,unique_dates,expected_calendar_days,missing_calendar_days,invalid_dates
0,train,date,2013-01-01,2017-08-15,1684,1688,4,0
1,test,date,2017-08-16,2017-08-31,16,16,0,0
2,oil,date,2013-01-01,2017-08-31,1218,1704,486,0
3,holidays,date,2012-03-02,2017-12-26,312,2126,1814,0
4,transactions,date,2013-01-01,2017-08-15,1682,1688,6,0


,train_min_date,train_max_date,test_min_date,test_max_date,train_unique_dates,test_unique_dates,test_horizon_days,test_starts_after_train,train_test_overlap_days
0,2013-01-01,2017-08-15,2017-08-16,2017-08-31,1684,16,16,True,0


# 11.Calendar Gaps Detail

This section lists the exact missing calendar dates for time-based datasets.

Not all datasets are expected to have one row per calendar day. For example, `holidays_events.csv` only contains event dates, so calendar gaps are expected and should not be treated as missing data.


In [8]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


datasets_to_check_for_calendar_gaps = {
    "train": "date",
    "test": "date",
    "oil": "date",
    "transactions": "date",
}

calendar_gap_rows = []

for dataset_name, date_column in datasets_to_check_for_calendar_gaps.items():
    df = raw_datasets[dataset_name].copy()
    dates = pd.to_datetime(df[date_column], errors="coerce").dropna()

    min_date = dates.min()
    max_date = dates.max()

    full_calendar = pd.date_range(start=min_date, end=max_date, freq="D")
    available_dates = pd.DatetimeIndex(dates.unique())

    missing_dates = sorted(set(full_calendar) - set(available_dates))

    for missing_date in missing_dates:
        calendar_gap_rows.append(
            {
                "dataset": dataset_name,
                "missing_date": missing_date,
            }
        )


calendar_gaps_df = pd.DataFrame(calendar_gap_rows)

if calendar_gaps_df.empty:
    calendar_gaps_df = pd.DataFrame(columns=["dataset", "missing_date"])

display(calendar_gaps_df)

calendar_gap_summary_df = (
    calendar_gaps_df.groupby("dataset", as_index=False)
    .agg(missing_calendar_days=("missing_date", "count"))
    .sort_values("dataset")
)

display(calendar_gap_summary_df)

calendar_gaps_df.to_csv(
    DATA_QUALITY_DIR / "calendar_gaps_detail.csv",
    index=False,
)

calendar_gap_summary_df.to_csv(
    DATA_QUALITY_DIR / "calendar_gaps_summary.csv",
    index=False,
)


,dataset,missing_date
0,train,2013-12-25
1,train,2014-12-25
2,train,2015-12-25
3,train,2016-12-25
4,oil,2013-01-05
...,...,...
491,transactions,2014-12-25
492,transactions,2015-12-25
493,transactions,2016-01-01
494,transactions,2016-01-03


,dataset,missing_calendar_days
0,oil,486
1,train,4
2,transactions,6


Calendar gaps are expected in oil and holidays-related time series. 
The train dataset only misses Christmas dates, which is consistent with store closure behavior.
Oil requires daily calendar completion before it can be safely joined to train and test.


# 12. Key Uniqueness and Duplicate Audit

This section validates the expected logical keys for each dataset.

Key uniqueness is critical beacuse duplicated keys can create row multiplication during joins, distort sales aggregation, and introduce incorrect trainig examples.

The `holidays` table is treated differently beacuse multiple events can occur on the same date. Therfore, duplicated in `holidays` are expected and must be handled later trouht aggregation before joining. 


In [9]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


key_rules = [
    {
        "dataset": "train",
        "key_columns": ["date", "store_nbr", "family"],
        "unique_required": True,
        "reason": "Main historical forecasting grain",
    },
    {
        "dataset": "test",
        "key_columns": ["date", "store_nbr", "family"],
        "unique_required": True,
        "reason": "Future forecasting grain",
    },
    {
        "dataset": "stores",
        "key_columns": ["store_nbr"],
        "unique_required": True,
        "reason": "Store master table",
    },
    {
        "dataset": "oil",
        "key_columns": ["date"],
        "unique_required": True,
        "reason": "Daily external variable",
    },
    {
        "dataset": "transactions",
        "key_columns": ["date", "store_nbr"],
        "unique_required": True,
        "reason": "Historical store-level activity",
    },
    {
        "dataset": "sample_submission",
        "key_columns": ["id"],
        "unique_required": True,
        "reason": "Submission identifier",
    },
    {
        "dataset": "holidays",
        "key_columns": ["date"],
        "unique_required": False,
        "reason": "Multiple events can occur on the same date",
    },
]


key_audit_rows = []
duplicate_key_detail_frames = []

for rule in key_rules:
    dataset_name = rule["dataset"]
    key_columns = rule["key_columns"]
    unique_required = rule["unique_required"]

    df = raw_datasets[dataset_name].copy()

    missing_key_columns = [column for column in key_columns if column not in df.columns]

    if missing_key_columns:
        raise ValueError(
            f"Dataset '{dataset_name}' is missing key columns: "
            + ", ".join(missing_key_columns)
        )

    total_rows = len(df)
    unique_keys = df[key_columns].drop_duplicates().shape[0]

    duplicated_key_mask = df.duplicated(subset=key_columns, keep=False)
    duplicated_key_rows = int(duplicated_key_mask.sum())

    duplicate_key_groups = (
        df.loc[duplicated_key_mask, key_columns].drop_duplicates().shape[0]
    )

    key_is_unique = duplicated_key_rows == 0

    key_audit_rows.append(
        {
            "dataset": dataset_name,
            "key_columns": " + ".join(key_columns),
            "total_rows": total_rows,
            "unique_keys": unique_keys,
            "duplicated_key_rows": duplicated_key_rows,
            "duplicate_key_groups": duplicate_key_groups,
            "unique_required": unique_required,
            "key_is_unique": key_is_unique,
            "validation_status": (
                "pass" if key_is_unique or not unique_required else "fail"
            ),
            "reason": rule["reason"],
        }
    )

    if duplicated_key_rows > 0:
        duplicate_detail = (
            df.loc[duplicated_key_mask, key_columns]
            .value_counts()
            .reset_index(name="duplicate_count")
        )
        duplicate_detail.insert(0, "dataset", dataset_name)
        duplicate_key_detail_frames.append(duplicate_detail)


key_uniqueness_audit_df = pd.DataFrame(key_audit_rows)

if duplicate_key_detail_frames:
    duplicate_key_details_df = pd.concat(
        duplicate_key_detail_frames,
        ignore_index=True,
    )
else:
    duplicate_key_details_df = pd.DataFrame(columns=["dataset", "duplicate_count"])


display(key_uniqueness_audit_df)
display(duplicate_key_details_df)


key_uniqueness_audit_df.to_csv(
    DATA_QUALITY_DIR / "key_uniqueness_audit.csv",
    index=False,
)

duplicate_key_details_df.to_csv(
    DATA_QUALITY_DIR / "duplicate_key_details.csv",
    index=False,
)


failed_key_validations = key_uniqueness_audit_df.loc[
    (key_uniqueness_audit_df["unique_required"])
    & (key_uniqueness_audit_df["validation_status"] == "fail"),
    "dataset",
].tolist()

if failed_key_validations:
    raise ValueError(
        "Key uniqueness validation failed for datasets: "
        + ", ".join(failed_key_validations)
    )


,dataset,key_columns,total_rows,unique_keys,duplicated_key_rows,duplicate_key_groups,unique_required,key_is_unique,validation_status,reason
0,train,date + store_nbr + family,3000888,3000888,0,0,True,True,pass,Main historical forecasting grain
1,test,date + store_nbr + family,28512,28512,0,0,True,True,pass,Future forecasting grain
2,stores,store_nbr,54,54,0,0,True,True,pass,Store master table
3,oil,date,1218,1218,0,0,True,True,pass,Daily external variable
4,transactions,date + store_nbr,83488,83488,0,0,True,True,pass,Historical store-level activity
5,sample_submission,id,28512,28512,0,0,True,True,pass,Submission identifier
6,holidays,date,350,312,69,31,False,False,pass,Multiple events can occur on the same date


,dataset,date,duplicate_count
0,holidays,2014-06-25,4
1,holidays,2012-06-25,3
2,holidays,2013-06-25,3
3,holidays,2015-06-25,3
4,holidays,2016-06-25,3
5,holidays,2017-06-25,3
6,holidays,2012-07-03,2
7,holidays,2012-12-22,2
8,holidays,2012-12-24,2
9,holidays,2012-12-31,2


## Key audit Summary

All critical datasets passed the key uniqueness validation.

The forecasting grain is confirmed as:

`data + store_nbr + family`

The holidays dataset contains multiple records for some dates.
This is expected because several events can occur on the same day.
However, this table must be aggregated before, joining it with `train` or `test`
to avoid row multiplication. 


# 13.Missing Values Audit

This section audits missing values across all raw datasets.

Missing values are evaluated at column level and classified according to their potential impact on the forecasting workflow.
Critical columns such as keys and the target variable must not contain missing values.


In [10]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ",".join(missing_objects)
    )

critical_columns_by_dataset = {
    "train": ["id", "date", "store_nbr", "family", "sales"],
    "test": ["id", "date", "store_nbr", "family"],
    "stores": ["store_nbr"],
    "holidays": ["date"],
    "transactions": ["date", "store_nbr", "transaction"],
    "oil": ["date"],
    "sample_submission": ["id", "sales"],
}

missing_values_rows = []

for dataset_name, df in raw_datasets.items():
    total_rows = len(df)
    critical_columns = critical_columns_by_dataset.get(dataset_name, [])

    for column_name in df.columns:
        missing_values = int(df[column_name].isna().sum())
        missing_pct = (
            round(missing_values / total_rows * 100, 4) if total_rows > 0 else 0
        )
        is_critical_column = column_name in critical_columns

    missing_values_rows.append(
        {
            "dataset": dataset_name,
            "column_name": column_name,
            "total_rows": total_rows,
            "missing_values": missing_values,
            "missing_pct": missing_pct,
            "is_critical_column": is_critical_column,
            "requires_action": missing_values > 0,
            "critical_issue": missing_values > 0 and is_critical_column,
        }
    )

missing_values_audit_df = pd.DataFrame(missing_values_rows)

missing_values_summary_df = missing_values_audit_df.groupby(
    "dataset", as_index=False
).agg(
    total_missing_values=("missing_values", "sum"),
    columns_with_missing_values=("requires_action", "sum"),
    critical_issues=("critical_issue", "sum"),
)

display(missing_values_summary_df)
display(
    missing_values_audit_df.loc[
        missing_values_audit_df["missing_values"] > 0
    ].sort_values(["dataset", "missing_values"], ascending=[True, False])
)

missing_values_audit_df.to_csv(
    DATA_QUALITY_DIR / "missing_values_audit.csv",
    index=False,
)

missing_values_summary_df.to_csv(
    DATA_QUALITY_DIR / "missing_values_summary.csv",
    index=False,
)

critical_missing_values = missing_values_audit_df.loc[
    missing_values_audit_df["critical_issue"],
    ["dataset", "column_name", "missing_values"],
]

if not critical_missing_values.empty:
    display(critical_missing_values)

    raise ValueError(
        "Critical missing values detected. "
        "Review the affected datasets before continuing. "
    )


,dataset,total_missing_values,columns_with_missing_values,critical_issues
0,holidays,0,0,0
1,oil,43,1,0
2,sample_submission,0,0,0
3,stores,0,0,0
4,test,0,0,0
5,train,0,0,0
6,transactions,0,0,0


,dataset,column_name,total_rows,missing_values,missing_pct,is_critical_column,requires_action,critical_issue
3,oil,dcoilwtico,1218,43,3.5304,False,True,False


# 14.Numeric Values Audit.

This section audits numeric columns across the raw datasets. 
The goal is to detect impossible or suspicious values such as negative sales, negative promotions, negative transactions, or invalid oil prices. 


In [11]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


numeric_audit_rows = []

for dataset_name, df in raw_datasets.items():
    numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()

    for column_name in numeric_columns:
        series = df[column_name]

        numeric_audit_rows.append(
            {
                "dataset": dataset_name,
                "column_name": column_name,
                "dtype": str(series.dtype),
                "total_rows": len(series),
                "missing_values": int(series.isna().sum()),
                "min_value": series.min(),
                "max_value": series.max(),
                "mean_value": series.mean(),
                "median_value": series.median(),
                "zero_values": int((series == 0).sum()),
                "zero_pct": (
                    round((series == 0).sum() / len(series) * 100, 4)
                    if len(series) > 0
                    else 0
                ),
                "negative_values": int((series < 0).sum()),
            }
        )


numeric_values_audit_df = pd.DataFrame(numeric_audit_rows)

display(numeric_values_audit_df)

numeric_values_audit_df.to_csv(
    DATA_QUALITY_DIR / "numeric_values_audit.csv",
    index=False,
)


,dataset,column_name,dtype,total_rows,missing_values,min_value,max_value,mean_value,median_value,zero_values,zero_pct,negative_values
0,train,id,int64,3000888,0,0.00,3000887.00,1.500444e+06,1500443.50,1,0.0000,0
1,train,store_nbr,int64,3000888,0,1.00,54.00,2.750000e+01,27.50,0,0.0000,0
2,train,sales,float64,3000888,0,0.00,124717.00,3.577757e+02,11.00,939130,31.2951,0
3,train,onpromotion,int64,3000888,0,0.00,741.00,2.602770e+00,0.00,2389559,79.6284,0
4,test,id,int64,28512,0,3000888.00,3029399.00,3.015144e+06,3015143.50,0,0.0000,0
5,test,store_nbr,int64,28512,0,1.00,54.00,2.750000e+01,27.50,0,0.0000,0
6,test,onpromotion,int64,28512,0,0.00,646.00,6.965383e+00,0.00,15943,55.9168,0
7,stores,store_nbr,int64,54,0,1.00,54.00,2.750000e+01,27.50,0,0.0000,0
8,stores,cluster,int64,54,0,1.00,17.00,8.481481e+00,8.50,0,0.0000,0
9,oil,dcoilwtico,float64,1218,43,26.19,110.62,6.771437e+01,53.19,0,0.0000,0


In [12]:
invalid_numeric_rules = [
    {
        "dataset": "train",
        "column_name": "sales",
        "rule": "sales must be greater than or equal to 0",
        "invalid_count": int((raw_datasets["train"]["sales"] < 0).sum()),
    },
    {
        "dataset": "train",
        "column_name": "onpromotion",
        "rule": "onpromotion must be greater than or equal to 0",
        "invalid_count": int((raw_datasets["train"]["onpromotion"] < 0).sum()),
    },
    {
        "dataset": "test",
        "column_name": "onpromotion",
        "rule": "onpromotion must be greater than or equal to 0",
        "invalid_count": int((raw_datasets["test"]["onpromotion"] < 0).sum()),
    },
    {
        "dataset": "transactions",
        "column_name": "transactions",
        "rule": "transactions must be greater than or equal to 0",
        "invalid_count": int((raw_datasets["transactions"]["transactions"] < 0).sum()),
    },
    {
        "dataset": "oil",
        "column_name": "dcoilwtico",
        "rule": "oil price must be greater than 0 when available",
        "invalid_count": int((raw_datasets["oil"]["dcoilwtico"].dropna() <= 0).sum()),
    },
]

invalid_numeric_rules_df = pd.DataFrame(invalid_numeric_rules)

display(invalid_numeric_rules_df)

invalid_numeric_rules_df.to_csv(
    DATA_QUALITY_DIR / "invalid_numeric_rules.csv",
    index=False,
)

failed_numeric_rules = invalid_numeric_rules_df.loc[
    invalid_numeric_rules_df["invalid_count"] > 0,
    ["dataset", "column_name", "invalid_count"],
]

if not failed_numeric_rules.empty:
    display(failed_numeric_rules)

    raise ValueError(
        "Invalid numeric values detected. "
        "Review the affected variables before continuing."
    )


,dataset,column_name,rule,invalid_count
0,train,sales,sales must be greater than or equal to 0,0
1,train,onpromotion,onpromotion must be greater than or equal to 0,0
2,test,onpromotion,onpromotion must be greater than or equal to 0,0
3,transactions,transactions,transactions must be greater than or equal to 0,0
4,oil,dcoilwtico,oil price must be greater than 0 when available,0


# 15. Categorical Values Audit.

This section audits categorical variables across the raw datasets.

The goal is to validate category consitency, detect unexpected category mismatches, and indetify variables that may be relevant for downstream forecasting. 


In [13]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )

categorical_audit_rows = []

for dataset_name, df in raw_datasets.items():
    categorical_columns = df.select_dtypes(
        include=["object", "string", "bool"]
    ).columns.tolist()

    for column_name in categorical_columns:
        unique_values = df[column_name].dropna().unique()
        unique_values_sorted = sorted([str(value) for value in unique_values])

        categorical_audit_rows.append(
            {
                "dataset": dataset_name,
                "columne_name": column_name,
                "dtype": str(df[column_name].dtype),
                "total_rows": len(df),
                "missing_values": int(df[column_name].isna().sum()),
                "unique_values": len(unique_values),
                "sample_values": ",".join(unique_values_sorted[:10]),
            }
        )

categorical_values_audit_df = pd.DataFrame(categorical_audit_rows)

display(categorical_values_audit_df)

categorical_values_audit_df.to_csv(
    DATA_QUALITY_DIR / "categorical_values_audit_df.csv",
    index=False,
)


,dataset,columne_name,dtype,total_rows,missing_values,unique_values,sample_values
0,train,family,str,3000888,0,33,"AUTOMOTIVE,BABY CARE,BEAUTY,BEVERAGES,BOOKS,BR..."
1,test,family,str,28512,0,33,"AUTOMOTIVE,BABY CARE,BEAUTY,BEVERAGES,BOOKS,BR..."
2,stores,city,str,54,0,22,"Ambato,Babahoyo,Cayambe,Cuenca,Daule,El Carmen..."
3,stores,state,str,54,0,16,"Azuay,Bolivar,Chimborazo,Cotopaxi,El Oro,Esmer..."
4,stores,type,str,54,0,5,"A,B,C,D,E"
5,holidays,type,str,350,0,6,"Additional,Bridge,Event,Holiday,Transfer,Work Day"
6,holidays,locale,str,350,0,3,"Local,National,Regional"
7,holidays,locale_name,str,350,0,24,"Ambato,Cayambe,Cotopaxi,Cuenca,Ecuador,El Carm..."
8,holidays,description,str,350,0,103,"Batalla de Pichincha,Black Friday,Cantonizacio..."
9,holidays,transferred,bool,350,0,2,"False,True"


In [14]:
train_families = set(raw_datasets["train"]["family"].unique())
test_families = set(raw_datasets["test"]["family"].unique())

families_only_in_train = sorted(train_families - test_families)
families_only_in_test = sorted(test_families - train_families)

store_types = sorted(raw_datasets["stores"]["type"].unique())
store_cities = sorted(raw_datasets["stores"]["city"].unique())
store_states = sorted(raw_datasets["stores"]["state"].unique())
store_clusters = sorted(raw_datasets["stores"]["cluster"].unique())

holiday_types = sorted(raw_datasets["holidays"]["type"].unique())
holiday_locales = sorted(raw_datasets["holidays"]["locale"].unique())
holiday_transferred_values = sorted(raw_datasets["holidays"]["transferred"].unique())


categorical_consistency_checks = [
    {
        "check_name": "families_only_in_train",
        "issue_count": len(families_only_in_train),
        "values": families_only_in_train,
        "status": "pass" if len(families_only_in_train) == 0 else "review",
    },
    {
        "check_name": "families_only_in_test",
        "issue_count": len(families_only_in_test),
        "values": families_only_in_test,
        "status": "pass" if len(families_only_in_test) == 0 else "fail",
    },
    {
        "check_name": "store_types",
        "issue_count": 0,
        "values": store_types,
        "status": "info",
    },
    {
        "check_name": "store_cities",
        "issue_count": 0,
        "values": store_cities,
        "status": "info",
    },
    {
        "check_name": "store_states",
        "issue_count": 0,
        "values": store_states,
        "status": "info",
    },
    {
        "check_name": "store_clusters",
        "issue_count": 0,
        "values": store_clusters,
        "status": "info",
    },
    {
        "check_name": "holiday_types",
        "issue_count": 0,
        "values": holiday_types,
        "status": "info",
    },
    {
        "check_name": "holiday_locales",
        "issue_count": 0,
        "values": holiday_locales,
        "status": "info",
    },
    {
        "check_name": "holiday_transferred_values",
        "issue_count": 0,
        "values": holiday_transferred_values,
        "status": "info",
    },
]

categorical_consistency_checks_df = pd.DataFrame(categorical_consistency_checks)

display(categorical_consistency_checks_df)

categorical_consistency_checks_df.to_csv(
    DATA_QUALITY_DIR / "categorical_consistency_checks.csv",
    index=False,
)


failed_categorical_checks = categorical_consistency_checks_df.loc[
    categorical_consistency_checks_df["status"] == "fail",
    "check_name",
].tolist()

if failed_categorical_checks:
    raise ValueError(
        "Categorical consistency validation failed for checks: "
        + ", ".join(failed_categorical_checks)
    )


,check_name,issue_count,values,status
0,families_only_in_train,0,[],pass
1,families_only_in_test,0,[],pass
2,store_types,0,"[A, B, C, D, E]",info
3,store_cities,0,"[Ambato, Babahoyo, Cayambe, Cuenca, Daule, El ...",info
4,store_states,0,"[Azuay, Bolivar, Chimborazo, Cotopaxi, El Oro,...",info
5,store_clusters,0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",info
6,holiday_types,0,"[Additional, Bridge, Event, Holiday, Transfer,...",info
7,holiday_locales,0,"[Local, National, Regional]",info
8,holiday_transferred_values,0,"[False, True]",info


# Train/Test Consistency Audit.

This section validates the structural consistency between the training and test datasets.
For this forecasting task, the test set should contain the same store and product family combination observed in the training set. The sample sumission must also match the test identifiers.


In [15]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


train = raw_datasets["train"]
test = raw_datasets["test"]
sample_submission = raw_datasets["sample_submission"]


train_stores = set(train["store_nbr"].unique())
test_stores = set(test["store_nbr"].unique())

train_families = set(train["family"].unique())
test_families = set(test["family"].unique())

train_store_family = set(
    train[["store_nbr", "family"]].drop_duplicates().itertuples(index=False, name=None)
)

test_store_family = set(
    test[["store_nbr", "family"]].drop_duplicates().itertuples(index=False, name=None)
)

train_dates = pd.to_datetime(train["date"])
test_dates = pd.to_datetime(test["date"])

expected_test_rows = (
    test_dates.nunique() * test["store_nbr"].nunique() * test["family"].nunique()
)

sample_submission_ids = set(sample_submission["id"])
test_ids = set(test["id"])

consistency_checks = [
    {
        "check_name": "stores_only_in_train",
        "issue_count": len(train_stores - test_stores),
        "status": "pass" if len(train_stores - test_stores) == 0 else "review",
        "details": sorted(train_stores - test_stores),
    },
    {
        "check_name": "stores_only_in_test",
        "issue_count": len(test_stores - train_stores),
        "status": "pass" if len(test_stores - train_stores) == 0 else "fail",
        "details": sorted(test_stores - train_stores),
    },
    {
        "check_name": "families_only_in_train",
        "issue_count": len(train_families - test_families),
        "status": "pass" if len(train_families - test_families) == 0 else "review",
        "details": sorted(train_families - test_families),
    },
    {
        "check_name": "families_only_in_test",
        "issue_count": len(test_families - train_families),
        "status": "pass" if len(test_families - train_families) == 0 else "fail",
        "details": sorted(test_families - train_families),
    },
    {
        "check_name": "store_family_combinations_only_in_train",
        "issue_count": len(train_store_family - test_store_family),
        "status": (
            "pass" if len(train_store_family - test_store_family) == 0 else "review"
        ),
        "details": sorted(train_store_family - test_store_family)[:20],
    },
    {
        "check_name": "store_family_combinations_only_in_test",
        "issue_count": len(test_store_family - train_store_family),
        "status": (
            "pass" if len(test_store_family - train_store_family) == 0 else "fail"
        ),
        "details": sorted(test_store_family - train_store_family)[:20],
    },
    {
        "check_name": "test_has_expected_full_grid",
        "issue_count": abs(len(test) - expected_test_rows),
        "status": "pass" if len(test) == expected_test_rows else "fail",
        "details": {
            "actual_test_rows": len(test),
            "expected_test_rows": expected_test_rows,
            "unique_test_dates": test_dates.nunique(),
            "unique_test_stores": test["store_nbr"].nunique(),
            "unique_test_families": test["family"].nunique(),
        },
    },
    {
        "check_name": "sample_submission_ids_match_test_ids",
        "issue_count": len(sample_submission_ids.symmetric_difference(test_ids)),
        "status": ("pass" if sample_submission_ids == test_ids else "fail"),
        "details": {
            "test_ids": len(test_ids),
            "sample_submission_ids": len(sample_submission_ids),
            "ids_not_matching": len(
                sample_submission_ids.symmetric_difference(test_ids)
            ),
        },
    },
]

train_test_consistency_df = pd.DataFrame(consistency_checks)

display(train_test_consistency_df)

train_test_consistency_df.to_csv(
    DATA_QUALITY_DIR / "train_test_consistency_audit.csv",
    index=False,
)


failed_train_test_checks = train_test_consistency_df.loc[
    train_test_consistency_df["status"] == "fail",
    "check_name",
].tolist()

if failed_train_test_checks:
    raise ValueError(
        "Train/test consistency validation failed for checks: "
        + ", ".join(failed_train_test_checks)
    )


,check_name,issue_count,status,details
0,stores_only_in_train,0,pass,[]
1,stores_only_in_test,0,pass,[]
2,families_only_in_train,0,pass,[]
3,families_only_in_test,0,pass,[]
4,store_family_combinations_only_in_train,0,pass,[]
5,store_family_combinations_only_in_test,0,pass,[]
6,test_has_expected_full_grid,0,pass,"{'actual_test_rows': 28512, 'expected_test_row..."
7,sample_submission_ids_match_test_ids,0,pass,"{'test_ids': 28512, 'sample_submission_ids': 2..."


# Referential Integrity Checks

This section validates whether the datasets can be safely connected through their logical keys.

The goal is to identify missing references and potential join risks before creating clean base tables.


In [16]:
required_objects = ["raw_datasets", "pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


train = raw_datasets["train"]
test = raw_datasets["test"]
stores = raw_datasets["stores"]
oil = raw_datasets["oil"]
holidays = raw_datasets["holidays"]
transactions = raw_datasets["transactions"]


referential_checks = []


# Store references
store_master_ids = set(stores["store_nbr"].unique())

for dataset_name, df in {
    "train": train,
    "test": test,
    "transactions": transactions,
}.items():
    dataset_store_ids = set(df["store_nbr"].unique())
    missing_store_ids = sorted(dataset_store_ids - store_master_ids)

    referential_checks.append(
        {
            "check_name": f"{dataset_name}_stores_exist_in_store_master",
            "reference_type": "store_nbr",
            "issue_count": len(missing_store_ids),
            "status": "pass" if len(missing_store_ids) == 0 else "fail",
            "details": missing_store_ids,
        }
    )


# Oil date coverage
train_test_dates = set(pd.concat([train["date"], test["date"]]).dropna().dt.date)

oil_dates = set(oil["date"].dropna().dt.date)

dates_missing_in_raw_oil = sorted(train_test_dates - oil_dates)

referential_checks.append(
    {
        "check_name": "train_test_dates_available_in_raw_oil",
        "reference_type": "date",
        "issue_count": len(dates_missing_in_raw_oil),
        "status": "review" if len(dates_missing_in_raw_oil) > 0 else "pass",
        "details": dates_missing_in_raw_oil[:20],
    }
)


# Holidays date overlap
holiday_dates = set(holidays["date"].dropna().dt.date)

train_dates = set(train["date"].dropna().dt.date)
test_dates = set(test["date"].dropna().dt.date)

holiday_dates_in_train = sorted(train_dates & holiday_dates)
holiday_dates_in_test = sorted(test_dates & holiday_dates)

referential_checks.append(
    {
        "check_name": "holiday_dates_overlap_train",
        "reference_type": "date",
        "issue_count": len(holiday_dates_in_train),
        "status": "info",
        "details": {
            "holiday_dates_in_train": len(holiday_dates_in_train),
            "sample_dates": holiday_dates_in_train[:20],
        },
    }
)

referential_checks.append(
    {
        "check_name": "holiday_dates_overlap_test",
        "reference_type": "date",
        "issue_count": len(holiday_dates_in_test),
        "status": "info",
        "details": {
            "holiday_dates_in_test": len(holiday_dates_in_test),
            "sample_dates": holiday_dates_in_test[:20],
        },
    }
)


# Transactions historical coverage
transaction_keys = set(
    transactions[["date", "store_nbr"]]
    .assign(date=lambda df: df["date"].dt.date)
    .itertuples(index=False, name=None)
)

train_store_date_keys = set(
    train[["date", "store_nbr"]]
    .drop_duplicates()
    .assign(date=lambda df: df["date"].dt.date)
    .itertuples(index=False, name=None)
)

train_store_date_keys_missing_transactions = sorted(
    train_store_date_keys - transaction_keys
)

referential_checks.append(
    {
        "check_name": "train_store_date_keys_available_in_transactions",
        "reference_type": "date + store_nbr",
        "issue_count": len(train_store_date_keys_missing_transactions),
        "status": (
            "review" if len(train_store_date_keys_missing_transactions) > 0 else "pass"
        ),
        "details": train_store_date_keys_missing_transactions[:20],
    }
)


referential_integrity_df = pd.DataFrame(referential_checks)

display(referential_integrity_df)

referential_integrity_df.to_csv(
    DATA_QUALITY_DIR / "referential_integrity_checks.csv",
    index=False,
)


failed_referential_checks = referential_integrity_df.loc[
    referential_integrity_df["status"] == "fail",
    "check_name",
].tolist()

if failed_referential_checks:
    raise ValueError(
        "Referential integrity validation failed for checks: "
        + ", ".join(failed_referential_checks)
    )


,check_name,reference_type,issue_count,status,details
0,train_stores_exist_in_store_master,store_nbr,0,pass,[]
1,test_stores_exist_in_store_master,store_nbr,0,pass,[]
2,transactions_stores_exist_in_store_master,store_nbr,0,pass,[]
3,train_test_dates_available_in_raw_oil,date,485,review,"[2013-01-05, 2013-01-06, 2013-01-12, 2013-01-1..."
4,holiday_dates_overlap_train,date,252,info,"{'holiday_dates_in_train': 252, 'sample_dates'..."
5,holiday_dates_overlap_test,date,1,info,"{'holiday_dates_in_test': 1, 'sample_dates': [..."
6,train_store_date_keys_available_in_transactions,date + store_nbr,7448,review,"[(2013-01-01, 1), (2013-01-01, 2), (2013-01-01..."


# Join Risk Assessment

This section documents the main join risks identified during the data audit.

The goal is to define which tables can be safely joined, which tables require preprocessing, and which variables must be handled carefully to avoid row multiplication or data leakage.

| Table | Join Key | Join Risk | Decision |
|---|---|---|---|
| `stores` | `store_nbr` | Low | Safe to join with `train` and `test`. `store_nbr` is unique in the store master table. |
| `oil` | `date` | Medium | Raw oil does not contain every calendar date and has missing price values. A complete daily oil table must be created before joining. |
| `holidays` | `date` | High | Multiple events can occur on the same date. The table must be aggregated by date before joining to avoid row multiplication. |
| `transactions` | `date + store_nbr` | High | Transactions are historical and not available for the test period. This table should be kept as an auxiliary historical dataset and not joined directly into the test base. |
| `sample_submission` | `id` | Low | Safe for validation only. It matches the test identifiers. |

## Join Decisions

The clean base tables created later in this notebook will include only joins that are safe and reproducible.

The recommended base joins are:

```text
train/test
+ stores_clean
+ oil_daily_clean
+ holidays_daily_clean```


The transactions table will be cleaned and saved separately, but it will not be included in the default modeling base tables at this stage.

Business Impact

Incorrect joins can create duplicated sales records, distorted demand patterns, and unreliable forecasts.

For a replenishment use case, this is especially risky because duplicated demand or leaked future information could lead to incorrect stock decisions.


# 19.Cleaning Decisions

This section documents the cleaning decisions that will be applied before generating reusable datasets.

The goal is to keep the cleaning process explicit, reproducible, and aligned with the forecasting use case.


In [17]:
required_objects = ["pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


cleaning_decisions = [
    {
        "area": "Raw data",
        "decision": "Never overwrite raw files",
        "reason": "Raw files are the source of truth and must remain unchanged.",
        "output_impact": "All generated files will be saved under data/interim, data/processed, or reports.",
    },
    {
        "area": "Train and test",
        "decision": "Keep train and test separated",
        "reason": "Keeping them separated reduces the risk of data leakage.",
        "output_impact": "train_clean and test_clean will be created separately.",
    },
    {
        "area": "Forecasting grain",
        "decision": "Use date + store_nbr + family as the forecasting grain",
        "reason": "This is the unique logical key available in both train and test.",
        "output_impact": "All base tables must preserve this grain.",
    },
    {
        "area": "Product metadata",
        "decision": "Use family as the available product-level variable",
        "reason": "The dataset does not include a separate item-level metadata table.",
        "output_impact": "No item-level joins will be created.",
    },
    {
        "area": "Oil",
        "decision": "Create a complete daily oil table and impute missing prices",
        "reason": "Raw oil data does not contain every calendar date and includes null price values.",
        "output_impact": "oil_daily_clean will be created before joining with train or test.",
    },
    {
        "area": "Holidays",
        "decision": "Aggregate holidays by date before joining",
        "reason": "Multiple holiday events can occur on the same date, creating row multiplication if joined directly.",
        "output_impact": "holidays_daily_clean will be created for safe joins.",
    },
    {
        "area": "Transactions",
        "decision": "Clean and save transactions separately",
        "reason": "Transactions are historical and are not available for the future test period.",
        "output_impact": "transactions_clean will be saved, but not included in the default train/test base tables.",
    },
    {
        "area": "Sales target",
        "decision": "Do not transform sales in this notebook",
        "reason": "Target transformations belong to feature engineering or modeling notebooks.",
        "output_impact": "sales will remain unchanged in train_clean and train_base.",
    },
    {
        "area": "Missing values",
        "decision": "Only handle missing values where required for safe joins",
        "reason": "The only detected missing values are in the auxiliary oil price column.",
        "output_impact": "Missing oil prices will be handled in oil_daily_clean.",
    },
    {
        "area": "Output format",
        "decision": "Use Parquet as the main output format",
        "reason": "Parquet is efficient, preserves data types better than CSV, and is suitable for reusable pipelines.",
        "output_impact": "Clean datasets will be saved as .parquet files.",
    },
]


cleaning_decisions_df = pd.DataFrame(cleaning_decisions)

display(cleaning_decisions_df)


cleaning_decisions_df.to_csv(
    DATA_QUALITY_DIR / "cleaning_decisions.csv",
    index=False,
)

with open(DATA_QUALITY_DIR / "cleaning_decisions.json", "w", encoding="utf-8") as file:
    json.dump(cleaning_decisions, file, indent=4, ensure_ascii=False)


,area,decision,reason,output_impact
0,Raw data,Never overwrite raw files,Raw files are the source of truth and must rem...,All generated files will be saved under data/i...
1,Train and test,Keep train and test separated,Keeping them separated reduces the risk of dat...,train_clean and test_clean will be created sep...
2,Forecasting grain,Use date + store_nbr + family as the forecasti...,This is the unique logical key available in bo...,All base tables must preserve this grain.
3,Product metadata,Use family as the available product-level vari...,The dataset does not include a separate item-l...,No item-level joins will be created.
4,Oil,Create a complete daily oil table and impute m...,Raw oil data does not contain every calendar d...,oil_daily_clean will be created before joining...
5,Holidays,Aggregate holidays by date before joining,Multiple holiday events can occur on the same ...,holidays_daily_clean will be created for safe ...
6,Transactions,Clean and save transactions separately,Transactions are historical and are not availa...,"transactions_clean will be saved, but not incl..."
7,Sales target,Do not transform sales in this notebook,Target transformations belong to feature engin...,sales will remain unchanged in train_clean and...
8,Missing values,Only handle missing values where required for ...,The only detected missing values are in the au...,Missing oil prices will be handled in oil_dail...
9,Output format,Use Parquet as the main output format,"Parquet is efficient, preserves data types bet...",Clean datasets will be saved as .parquet files.


# 20.Clean Oil Table

This section creates a complete daily oil price table.

The raw oil dataset does not contain every calendar date and includes missing `dcoilwtico` values. For safe joins with `train` and `test`, the oil table must be expanded to a complete daily calendar and missing prices must be imputed.

The imputation strategy is:

1. Create a complete daily calendar covering the full `train` and `test` date range.
2. Join the raw oil prices by date.
3. Forward-fill missing oil prices.
4. Backward-fill any remaining initial missing values.
5. Keep flags indicating whether the original oil row was available and whether the price was imputed.


In [18]:
required_objects = ["raw_datasets", "pd", "INTERIM_DIR", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


train_dates = pd.to_datetime(raw_datasets["train"]["date"])
test_dates = pd.to_datetime(raw_datasets["test"]["date"])

calendar_start = min(train_dates.min(), test_dates.min())
calendar_end = max(train_dates.max(), test_dates.max())

daily_calendar = pd.DataFrame(
    {
        "date": pd.date_range(
            start=calendar_start,
            end=calendar_end,
            freq="D",
        )
    }
)


oil_raw = raw_datasets["oil"].copy()
oil_raw["date"] = pd.to_datetime(oil_raw["date"])

oil_daily_clean = daily_calendar.merge(
    oil_raw,
    on="date",
    how="left",
)

raw_oil_dates = set(oil_raw["date"])

oil_daily_clean["oil_row_exists_in_raw"] = oil_daily_clean["date"].isin(raw_oil_dates)
oil_daily_clean["dcoilwtico_missing_before_imputation"] = oil_daily_clean[
    "dcoilwtico"
].isna()

oil_daily_clean["dcoilwtico"] = oil_daily_clean["dcoilwtico"].ffill().bfill()

oil_daily_clean["dcoilwtico_was_imputed"] = oil_daily_clean[
    "dcoilwtico_missing_before_imputation"
]


oil_clean_validation = {
    "calendar_start": oil_daily_clean["date"].min(),
    "calendar_end": oil_daily_clean["date"].max(),
    "total_days": len(oil_daily_clean),
    "missing_prices_after_imputation": int(oil_daily_clean["dcoilwtico"].isna().sum()),
    "raw_rows_available": int(oil_daily_clean["oil_row_exists_in_raw"].sum()),
    "calendar_rows_created": int((~oil_daily_clean["oil_row_exists_in_raw"]).sum()),
    "prices_imputed": int(oil_daily_clean["dcoilwtico_was_imputed"].sum()),
    "min_oil_price": oil_daily_clean["dcoilwtico"].min(),
    "max_oil_price": oil_daily_clean["dcoilwtico"].max(),
}

oil_clean_validation_df = pd.DataFrame([oil_clean_validation])

display(oil_clean_validation_df)
display(oil_daily_clean.head())
display(oil_daily_clean.tail())


oil_daily_clean.to_parquet(
    INTERIM_DIR / "oil_daily_clean.parquet",
    index=False,
)

oil_clean_validation_df.to_csv(
    DATA_QUALITY_DIR / "oil_clean_validation.csv",
    index=False,
)


if oil_clean_validation["missing_prices_after_imputation"] > 0:
    raise ValueError("Oil cleaning failed: missing prices remain after imputation.")


,calendar_start,calendar_end,total_days,missing_prices_after_imputation,raw_rows_available,calendar_rows_created,prices_imputed,min_oil_price,max_oil_price
0,2013-01-01,2017-08-31,1704,0,1218,486,529,26.19,110.62


,date,dcoilwtico,oil_row_exists_in_raw,dcoilwtico_missing_before_imputation,dcoilwtico_was_imputed
0,2013-01-01,93.14,True,True,True
1,2013-01-02,93.14,True,False,False
2,2013-01-03,92.97,True,False,False
3,2013-01-04,93.12,True,False,False
4,2013-01-05,93.12,False,True,True


,date,dcoilwtico,oil_row_exists_in_raw,dcoilwtico_missing_before_imputation,dcoilwtico_was_imputed
1699,2017-08-27,47.65,False,True,True
1700,2017-08-28,46.40,True,False,False
1701,2017-08-29,46.46,True,False,False
1702,2017-08-30,45.96,True,False,False
1703,2017-08-31,47.26,True,False,False


# 21.Clean Holidays Table

This section creates a daily aggregated holidays table.

The raw `holidays_events` dataset can contain multiple events on the same date. A direct join with `train` or `test` would multiply rows and distort sales records.

To avoid this, holidays are aggregated at daily level before being used in base tables.


In [ ]:
required_objects = ["raw_datasets", "pd", "INTERIM_DIR", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


train_dates = pd.to_datetime(raw_datasets["train"]["date"])
test_dates = pd.to_datetime(raw_datasets["test"]["date"])

calendar_start = min(train_dates.min(), test_dates.min())
calendar_end = max(train_dates.max(), test_dates.max())

daily_calendar = pd.DataFrame(
    {
        "date": pd.date_range(
            start=calendar_start,
            end=calendar_end,
            freq="D",
        )
    }
)


holidays_raw = raw_datasets["holidays"].copy()
holidays_raw["date"] = pd.to_datetime(holidays_raw["date"])

holidays_raw["type_clean"] = (
    holidays_raw["type"].str.lower().str.replace(" ", "_", regex=False)
)

holidays_raw["locale_clean"] = (
    holidays_raw["locale"].str.lower().str.replace(" ", "_", regex=False)
)


holiday_daily_base = holidays_raw.groupby("date", as_index=False).agg(
    calendar_event_count=("description", "count"),
    transferred_event_count=("transferred", "sum"),
    unique_locale_names=("locale_name", "nunique"),
)

holiday_daily_base["non_transferred_event_count"] = (
    holiday_daily_base["calendar_event_count"]
    - holiday_daily_base["transferred_event_count"]
)


holiday_type_counts = (
    pd.crosstab(
        holidays_raw["date"],
        holidays_raw["type_clean"],
    )
    .add_prefix("holiday_type_")
    .reset_index()
)

holiday_locale_counts = (
    pd.crosstab(
        holidays_raw["date"],
        holidays_raw["locale_clean"],
    )
    .add_prefix("holiday_locale_")
    .reset_index()
)


holidays_daily_clean = (
    daily_calendar.merge(holiday_daily_base, on="date", how="left")
    .merge(holiday_type_counts, on="date", how="left")
    .merge(holiday_locale_counts, on="date", how="left")
)


count_columns = [
    column_name for column_name in holidays_daily_clean.columns if column_name != "date"
]

holidays_daily_clean[count_columns] = (
    holidays_daily_clean[count_columns].fillna(0).astype("int64")
)

holidays_daily_clean["is_calendar_event"] = (
    holidays_daily_clean["calendar_event_count"] > 0
)

holidays_daily_clean["has_non_transferred_event"] = (
    holidays_daily_clean["non_transferred_event_count"] > 0
)

holidays_daily_clean["has_transferred_event"] = (
    holidays_daily_clean["transferred_event_count"] > 0
)


holidays_clean_validation = {
    "calendar_start": holidays_daily_clean["date"].min(),
    "calendar_end": holidays_daily_clean["date"].max(),
    "total_days": len(holidays_daily_clean),
    "dates_with_events": int(holidays_daily_clean["is_calendar_event"].sum()),
    "dates_with_non_transferred_events": int(
        holidays_daily_clean["has_non_transferred_event"].sum()
    ),
    "dates_with_transferred_events": int(
        holidays_daily_clean["has_transferred_event"].sum()
    ),
    "max_events_on_same_date": int(holidays_daily_clean["calendar_event_count"].max()),
    "duplicated_dates_after_cleaning": int(
        holidays_daily_clean.duplicated(subset=["date"]).sum()
    ),
}

holidays_clean_validation_df = pd.DataFrame([holidays_clean_validation])

display(holidays_clean_validation_df)
display(holidays_daily_clean.head())
display(
    holidays_daily_clean.loc[holidays_daily_clean["calendar_event_count"] > 0].head(10)
)


holidays_daily_clean.to_parquet(
    INTERIM_DIR / "holidays_daily_clean.parquet",
    index=False,
)

holidays_clean_validation_df.to_csv(
    DATA_QUALITY_DIR / "holidays_clean_validation.csv",
    index=False,
)


if holidays_clean_validation["duplicated_dates_after_cleaning"] > 0:
    raise ValueError(
        "Holiday cleaning failed: duplicated dates remain after aggregation."
    )


,calendar_start,calendar_end,total_days,dates_with_events,dates_with_non_transferred_events,dates_with_transferred_events,max_events_on_same_date,duplicated_dates_after_cleaning
0,2013-01-01,2017-08-31,1704,257,248,9,4,0


,date,calendar_event_count,transferred_event_count,unique_locale_names,non_transferred_event_count,holiday_type_additional,holiday_type_bridge,holiday_type_event,holiday_type_holiday,holiday_type_transfer,holiday_type_work_day,holiday_locale_local,holiday_locale_national,holiday_locale_regional,is_calendar_event,has_non_transferred_event,has_transferred_event
0,2013-01-01,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
1,2013-01-02,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
2,2013-01-03,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
3,2013-01-04,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
4,2013-01-05,1,0,1,1,0,0,0,0,0,1,0,1,0,True,True,False


,date,calendar_event_count,transferred_event_count,unique_locale_names,non_transferred_event_count,holiday_type_additional,holiday_type_bridge,holiday_type_event,holiday_type_holiday,holiday_type_transfer,holiday_type_work_day,holiday_locale_local,holiday_locale_national,holiday_locale_regional,is_calendar_event,has_non_transferred_event,has_transferred_event
0,2013-01-01,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
4,2013-01-05,1,0,1,1,0,0,0,0,0,1,0,1,0,True,True,False
11,2013-01-12,1,0,1,1,0,0,0,0,0,1,0,1,0,True,True,False
41,2013-02-11,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
42,2013-02-12,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
60,2013-03-02,1,0,1,1,0,0,0,1,0,0,1,0,0,True,True,False
90,2013-04-01,1,0,1,1,0,0,0,1,0,0,0,0,1,True,True,False
101,2013-04-12,1,0,1,1,0,0,0,1,0,0,1,0,0,True,True,False
103,2013-04-14,1,0,1,1,0,0,0,1,0,0,1,0,0,True,True,False
110,2013-04-21,1,0,1,1,0,0,0,1,0,0,1,0,0,True,True,False


# 22.Clean Stores Table

This section creates a clean store master table.

The `stores` table contains static store metadata and is safe to join with `train` and `test` using `store_nbr`.


In [ ]:
required_objects = ["raw_datasets", "pd", "INTERIM_DIR", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


stores_clean = raw_datasets["stores"].copy()

stores_clean = stores_clean.sort_values("store_nbr").reset_index(drop=True)

stores_clean["store_nbr"] = stores_clean["store_nbr"].astype("int64")
stores_clean["city"] = stores_clean["city"].astype("string")
stores_clean["state"] = stores_clean["state"].astype("string")
stores_clean["type"] = stores_clean["type"].astype("string")
stores_clean["cluster"] = stores_clean["cluster"].astype("int64")


stores_clean_validation = {
    "total_rows": len(stores_clean),
    "unique_stores": stores_clean["store_nbr"].nunique(),
    "duplicated_store_nbr": int(stores_clean.duplicated(subset=["store_nbr"]).sum()),
    "missing_values": int(stores_clean.isna().sum().sum()),
    "unique_cities": stores_clean["city"].nunique(),
    "unique_states": stores_clean["state"].nunique(),
    "unique_store_types": stores_clean["type"].nunique(),
    "unique_clusters": stores_clean["cluster"].nunique(),
}

stores_clean_validation_df = pd.DataFrame([stores_clean_validation])

display(stores_clean_validation_df)
display(stores_clean.head())


stores_clean.to_parquet(
    INTERIM_DIR / "stores_clean.parquet",
    index=False,
)

stores_clean_validation_df.to_csv(
    DATA_QUALITY_DIR / "stores_clean_validation.csv",
    index=False,
)


if stores_clean_validation["duplicated_store_nbr"] > 0:
    raise ValueError("Stores cleaning failed: duplicated store_nbr values detected.")

if stores_clean_validation["missing_values"] > 0:
    raise ValueError("Stores cleaning failed: missing values detected.")


,total_rows,unique_stores,duplicated_store_nbr,missing_values,unique_cities,unique_states,unique_store_types,unique_clusters
0,54,54,0,0,22,16,5,17


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


# 23.Clean Transactions Table

This section creates a clean transactions table.

The `transactions` dataset contains historical store-level activity by date. It is useful for analysis, but it should be handled carefully because future transactions are not available for the test period.

For this reason, the cleaned transactions table is saved separately and will not be included in the default train/test base tables at this stage.


In [ ]:
required_objects = ["raw_datasets", "pd", "INTERIM_DIR", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells." + ",".join(missing_objects)
    )

transactions_clean = raw_datasets["transactions"].copy()

transactions_clean["date"] = pd.to_datetime(transactions_clean["date"])
transactions_clean["store_nbr"] = transactions_clean["store_nbr"].astype("int64")
transactions_clean["transactions"] = transactions_clean["transactions"].astype("int64")


transactions_clean = transactions_clean.sort_values(["date", "store_nbr"]).reset_index(
    drop=True
)

transactions_clean_validation = {
    "total_rows": len(transactions_clean),
    "unique_dates": transactions_clean["date"].nunique(),
    "unique_stores": transactions_clean["store_nbr"].nunique(),
    "duplicated_date_store_keys": int(
        transactions_clean.duplicated(subset=["date", "store_nbr"]).sum()
    ),
    "missing_values": int(transactions_clean.isna().sum().sum()),
    "negative_transactions": int((transactions_clean["transactions"] < 0).sum()),
    "min_transactions": int(transactions_clean["transactions"].min()),
    "max_transactions": int(transactions_clean["transactions"].max()),
}

transactions_clean_validation_df = pd.DataFrame([transactions_clean_validation])

display(transactions_clean_validation_df)
display(transactions_clean.head())

transactions_clean.to_parquet(
    INTERIM_DIR / "transactions_clean.parquet",
    index=False,
)

transactions_clean_validation_df.to_csv(
    DATA_QUALITY_DIR / "transactions_clean_validation.csv",
    index=False,
)

if transactions_clean_validation["duplicated_date_store_keys"] > 0:
    raise ValueError(
        "Transactions cleaning failed: duplicated date + store_nbr keys detected"
    )

if transactions_clean_validation["missing_values"] > 0:
    raise ValueError("Transactions cleaning failed: missing values detected.")

if transactions_clean_validation["negative_transactions"] > 0:
    raise ValueError(
        "Transactions cleaning failed: negative transaction values detected"
    )


,total_rows,unique_dates,unique_stores,duplicated_date_store_keys,missing_values,negative_transactions,min_transactions,max_transactions
0,83488,1682,54,0,0,0,5,8359


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


# 24.Clean Train and Test Tables

This section creates clean versions of the train and test datasets.

Only structural cleaning is applied: date parsing, data type consistency, sorting, and validation of the expected forecasting grain.

The sales target is not transformed in this notebook.


In [ ]:
required_objects = ["raw_datasets", "pd", "INTERIM_DIR", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


train_clean = raw_datasets["train"].copy()
test_clean = raw_datasets["test"].copy()


# Type consistency
train_clean["id"] = train_clean["id"].astype("int64")
train_clean["date"] = pd.to_datetime(train_clean["date"])
train_clean["store_nbr"] = train_clean["store_nbr"].astype("int64")
train_clean["family"] = train_clean["family"].astype("string")
train_clean["sales"] = train_clean["sales"].astype("float64")
train_clean["onpromotion"] = train_clean["onpromotion"].astype("int64")

test_clean["id"] = test_clean["id"].astype("int64")
test_clean["date"] = pd.to_datetime(test_clean["date"])
test_clean["store_nbr"] = test_clean["store_nbr"].astype("int64")
test_clean["family"] = test_clean["family"].astype("string")
test_clean["onpromotion"] = test_clean["onpromotion"].astype("int64")


# Column order
train_clean = train_clean[["id", "date", "store_nbr", "family", "sales", "onpromotion"]]

test_clean = test_clean[["id", "date", "store_nbr", "family", "onpromotion"]]


# Row order
train_clean = train_clean.sort_values(["date", "store_nbr", "family"]).reset_index(
    drop=True
)

test_clean = test_clean.sort_values(["date", "store_nbr", "family"]).reset_index(
    drop=True
)


train_test_clean_validation = {
    "train_rows": len(train_clean),
    "test_rows": len(test_clean),
    "train_unique_ids": train_clean["id"].nunique(),
    "test_unique_ids": test_clean["id"].nunique(),
    "train_unique_forecasting_keys": train_clean[["date", "store_nbr", "family"]]
    .drop_duplicates()
    .shape[0],
    "test_unique_forecasting_keys": test_clean[["date", "store_nbr", "family"]]
    .drop_duplicates()
    .shape[0],
    "train_missing_values": int(train_clean.isna().sum().sum()),
    "test_missing_values": int(test_clean.isna().sum().sum()),
    "negative_sales": int((train_clean["sales"] < 0).sum()),
    "negative_train_onpromotion": int((train_clean["onpromotion"] < 0).sum()),
    "negative_test_onpromotion": int((test_clean["onpromotion"] < 0).sum()),
    "train_min_date": train_clean["date"].min(),
    "train_max_date": train_clean["date"].max(),
    "test_min_date": test_clean["date"].min(),
    "test_max_date": test_clean["date"].max(),
}

train_test_clean_validation_df = pd.DataFrame([train_test_clean_validation])

display(train_test_clean_validation_df)
display(train_clean.head())
display(test_clean.head())


train_clean.to_parquet(
    INTERIM_DIR / "train_clean.parquet",
    index=False,
)

test_clean.to_parquet(
    INTERIM_DIR / "test_clean.parquet",
    index=False,
)

train_test_clean_validation_df.to_csv(
    DATA_QUALITY_DIR / "train_test_clean_validation.csv",
    index=False,
)


if train_test_clean_validation["train_rows"] != len(raw_datasets["train"]):
    raise ValueError("Train cleaning failed: row count changed.")

if train_test_clean_validation["test_rows"] != len(raw_datasets["test"]):
    raise ValueError("Test cleaning failed: row count changed.")

if train_test_clean_validation["train_unique_ids"] != len(train_clean):
    raise ValueError("Train cleaning failed: duplicated id values detected.")

if train_test_clean_validation["test_unique_ids"] != len(test_clean):
    raise ValueError("Test cleaning failed: duplicated id values detected.")

if train_test_clean_validation["train_unique_forecasting_keys"] != len(train_clean):
    raise ValueError(
        "Train cleaning failed: duplicated date + store_nbr + family keys detected."
    )

if train_test_clean_validation["test_unique_forecasting_keys"] != len(test_clean):
    raise ValueError(
        "Test cleaning failed: duplicated date + store_nbr + family keys detected."
    )

if train_test_clean_validation["train_missing_values"] > 0:
    raise ValueError("Train cleaning failed: missing values detected.")

if train_test_clean_validation["test_missing_values"] > 0:
    raise ValueError("Test cleaning failed: missing values detected.")

if train_test_clean_validation["negative_sales"] > 0:
    raise ValueError("Train cleaning failed: negative sales values detected.")

if train_test_clean_validation["negative_train_onpromotion"] > 0:
    raise ValueError("Train cleaning failed: negative onpromotion values detected.")

if train_test_clean_validation["negative_test_onpromotion"] > 0:
    raise ValueError("Test cleaning failed: negative onpromotion values detected.")


,train_rows,test_rows,train_unique_ids,test_unique_ids,train_unique_forecasting_keys,test_unique_forecasting_keys,train_missing_values,test_missing_values,negative_sales,negative_train_onpromotion,negative_test_onpromotion,train_min_date,train_max_date,test_min_date,test_max_date
0,3000888,28512,3000888,28512,3000888,28512,0,0,0,0,0,2013-01-01,2017-08-15,2017-08-16,2017-08-31


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0


# Create Modeling Base Tables

This section creates the processed base tables that will be used in later EDA and modeling notebooks.

Only safe and reproducible joins are included:

text
train/test
+ stores_clean
+ oil_daily_clean
+ holidays_daily_clean

The transactions_clean table is not included in the default base tables because future transactions are not available for the test period.


In [ ]:
required_objects = [
    "train_clean",
    "test_clean",
    "stores_clean",
    "oil_daily_clean",
    "holidays_daily_clean",
    "pd",
    "PROCESSED_DIR",
    "DATA_QUALITY_DIR",
]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


stores_features = stores_clean.copy()
stores_features = stores_features.rename(columns={"type": "store_type"})

oil_features = oil_daily_clean.copy()
holidays_features = holidays_daily_clean.copy()


train_base = (
    train_clean.merge(
        stores_features,
        on="store_nbr",
        how="left",
        validate="many_to_one",
    )
    .merge(
        oil_features,
        on="date",
        how="left",
        validate="many_to_one",
    )
    .merge(
        holidays_features,
        on="date",
        how="left",
        validate="many_to_one",
    )
)

test_base = (
    test_clean.merge(
        stores_features,
        on="store_nbr",
        how="left",
        validate="many_to_one",
    )
    .merge(
        oil_features,
        on="date",
        how="left",
        validate="many_to_one",
    )
    .merge(
        holidays_features,
        on="date",
        how="left",
        validate="many_to_one",
    )
)


store_feature_columns = [
    "city",
    "state",
    "store_type",
    "cluster",
]

oil_feature_columns = [
    "dcoilwtico",
    "oil_row_exists_in_raw",
    "dcoilwtico_missing_before_imputation",
    "dcoilwtico_was_imputed",
]

holiday_feature_columns = [
    column_name for column_name in holidays_features.columns if column_name != "date"
]


base_table_validation = {
    "train_clean_rows": len(train_clean),
    "train_base_rows": len(train_base),
    "test_clean_rows": len(test_clean),
    "test_base_rows": len(test_base),
    "train_row_count_changed": len(train_clean) != len(train_base),
    "test_row_count_changed": len(test_clean) != len(test_base),
    "train_missing_store_features": int(
        train_base[store_feature_columns].isna().sum().sum()
    ),
    "test_missing_store_features": int(
        test_base[store_feature_columns].isna().sum().sum()
    ),
    "train_missing_oil_features": int(
        train_base[oil_feature_columns].isna().sum().sum()
    ),
    "test_missing_oil_features": int(test_base[oil_feature_columns].isna().sum().sum()),
    "train_missing_holiday_features": int(
        train_base[holiday_feature_columns].isna().sum().sum()
    ),
    "test_missing_holiday_features": int(
        test_base[holiday_feature_columns].isna().sum().sum()
    ),
    "train_columns": train_base.shape[1],
    "test_columns": test_base.shape[1],
    "transactions_included": False,
}

base_table_validation_df = pd.DataFrame([base_table_validation])

display(base_table_validation_df)
display(train_base.head())
display(test_base.head())


train_base.to_parquet(
    PROCESSED_DIR / "train_base.parquet",
    index=False,
)

test_base.to_parquet(
    PROCESSED_DIR / "test_base.parquet",
    index=False,
)

base_table_validation_df.to_csv(
    DATA_QUALITY_DIR / "base_table_validation.csv",
    index=False,
)


if base_table_validation["train_row_count_changed"]:
    raise ValueError("Base table creation failed: train row count changed after joins.")

if base_table_validation["test_row_count_changed"]:
    raise ValueError("Base table creation failed: test row count changed after joins.")

if base_table_validation["train_missing_store_features"] > 0:
    raise ValueError(
        "Base table creation failed: missing store features in train_base."
    )

if base_table_validation["test_missing_store_features"] > 0:
    raise ValueError("Base table creation failed: missing store features in test_base.")

if base_table_validation["train_missing_oil_features"] > 0:
    raise ValueError("Base table creation failed: missing oil features in train_base.")

if base_table_validation["test_missing_oil_features"] > 0:
    raise ValueError("Base table creation failed: missing oil features in test_base.")

if base_table_validation["train_missing_holiday_features"] > 0:
    raise ValueError(
        "Base table creation failed: missing holiday features in train_base."
    )

if base_table_validation["test_missing_holiday_features"] > 0:
    raise ValueError(
        "Base table creation failed: missing holiday features in test_base."
    )


,train_clean_rows,train_base_rows,test_clean_rows,test_base_rows,train_row_count_changed,test_row_count_changed,train_missing_store_features,test_missing_store_features,train_missing_oil_features,test_missing_oil_features,train_missing_holiday_features,test_missing_holiday_features,train_columns,test_columns,transactions_included
0,3000888,3000888,28512,28512,False,False,0,0,0,0,0,0,30,29,False


,id,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,dcoilwtico,oil_row_exists_in_raw,dcoilwtico_missing_before_imputation,dcoilwtico_was_imputed,calendar_event_count,transferred_event_count,unique_locale_names,non_transferred_event_count,holiday_type_additional,holiday_type_bridge,holiday_type_event,holiday_type_holiday,holiday_type_transfer,holiday_type_work_day,holiday_locale_local,holiday_locale_national,holiday_locale_regional,is_calendar_event,has_non_transferred_event,has_transferred_event
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,93.14,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,93.14,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,93.14,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,93.14,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,93.14,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False


,id,date,store_nbr,family,onpromotion,city,state,store_type,cluster,dcoilwtico,oil_row_exists_in_raw,dcoilwtico_missing_before_imputation,dcoilwtico_was_imputed,calendar_event_count,transferred_event_count,unique_locale_names,non_transferred_event_count,holiday_type_additional,holiday_type_bridge,holiday_type_event,holiday_type_holiday,holiday_type_transfer,holiday_type_work_day,holiday_locale_local,holiday_locale_national,holiday_locale_regional,is_calendar_event,has_non_transferred_event,has_transferred_event
0,3000888,2017-08-16,1,AUTOMOTIVE,0,Quito,Pichincha,D,13,46.8,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
1,3000889,2017-08-16,1,BABY CARE,0,Quito,Pichincha,D,13,46.8,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
2,3000890,2017-08-16,1,BEAUTY,2,Quito,Pichincha,D,13,46.8,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
3,3000891,2017-08-16,1,BEVERAGES,20,Quito,Pichincha,D,13,46.8,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
4,3000892,2017-08-16,1,BOOKS,0,Quito,Pichincha,D,13,46.8,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False


# 26.Data Quality Report Summary

This section validates and summarizes the data quality reports generated during the notebook.

These reports provide traceability for the audit, validation, and cleaning decisions applied before creating downstream datasets.


In [ ]:
required_objects = ["pd", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


expected_quality_reports = [
    "raw_files_inventory.csv",
    "expected_raw_files_validation.csv",
    "dataset_overview.csv",
    "column_overview.csv",
    "date_ranges_overview.csv",
    "schema_validation.csv",
    "date_range_audit.csv",
    "temporal_validation.csv",
    "calendar_gaps_detail.csv",
    "calendar_gaps_summary.csv",
    "key_uniqueness_audit.csv",
    "duplicate_key_details.csv",
    "missing_values_audit.csv",
    "missing_values_summary.csv",
    "numeric_values_audit.csv",
    "invalid_numeric_rules.csv",
    "categorical_values_audit.csv",
    "categorical_consistency_checks.csv",
    "train_test_consistency_audit.csv",
    "referential_integrity_checks.csv",
    "cleaning_decisions.csv",
    "cleaning_decisions.json",
    "oil_clean_validation.csv",
    "holidays_clean_validation.csv",
    "stores_clean_validation.csv",
    "transactions_clean_validation.csv",
    "train_test_clean_validation.csv",
    "base_table_validation.csv",
]

report_summary_rows = []

for file_name in expected_quality_reports:
    file_path = DATA_QUALITY_DIR / file_name
    file_exists = file_path.exists()

    report_summary_rows.append(
        {
            "file_name": file_name,
            "exists": file_exists,
            "file_size_kb": (
                round(file_path.stat().st_size / 1024, 2) if file_exists else None
            ),
            "modified_at": (
                pd.Timestamp.fromtimestamp(file_path.stat().st_mtime)
                if file_exists
                else None
            ),
            "path": str(file_path),
        }
    )


data_quality_report_summary_df = pd.DataFrame(report_summary_rows)

display(data_quality_report_summary_df)

data_quality_report_summary_df.to_csv(
    DATA_QUALITY_DIR / "data_quality_report_summary.csv",
    index=False,
)


missing_reports = data_quality_report_summary_df.loc[
    ~data_quality_report_summary_df["exists"],
    "file_name",
].tolist()

if missing_reports:
    raise FileNotFoundError(
        "Missing expected data quality reports: " + ", ".join(missing_reports)
    )


,file_name,exists,file_size_kb,modified_at,path
0,raw_files_inventory.csv,True,0.42,2026-05-22 10:16:13.780931,....
1,expected_raw_files_validation.csv,True,0.64,2026-05-22 10:16:13.797577,....
2,dataset_overview.csv,True,0.30,2026-05-22 10:16:22.253870,....
3,column_overview.csv,True,1.02,2026-05-22 10:16:22.264641,....
4,date_ranges_overview.csv,True,0.27,2026-05-22 10:16:22.264641,....
5,schema_validation.csv,True,0.87,2026-05-22 10:16:22.297103,....
6,date_range_audit.csv,True,0.35,2026-05-22 10:16:22.647779,....
7,temporal_validation.csv,True,0.22,2026-05-22 10:16:23.696947,....
8,calendar_gaps_detail.csv,True,7.83,2026-05-22 10:16:24.198411,....
9,calendar_gaps_summary.csv,True,0.06,2026-05-22 10:16:24.198411,....


# 27.Final Validation of Generated Outputs

This section validates that all expected clean datasets were generated successfully.

The goal is to confirm that the notebook produced reusable outputs for downstream EDA and modeling notebooks.


In [ ]:
required_objects = ["pd", "INTERIM_DIR", "PROCESSED_DIR", "DATA_QUALITY_DIR"]

missing_objects = [
    object_name for object_name in required_objects if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects from previous cells: " + ", ".join(missing_objects)
    )


expected_output_files = [
    {
        "file_name": "train_clean.parquet",
        "expected_location": "data/interim",
        "path": INTERIM_DIR / "train_clean.parquet",
        "expected_min_rows": 1,
    },
    {
        "file_name": "test_clean.parquet",
        "expected_location": "data/interim",
        "path": INTERIM_DIR / "test_clean.parquet",
        "expected_min_rows": 1,
    },
    {
        "file_name": "stores_clean.parquet",
        "expected_location": "data/interim",
        "path": INTERIM_DIR / "stores_clean.parquet",
        "expected_min_rows": 1,
    },
    {
        "file_name": "oil_daily_clean.parquet",
        "expected_location": "data/interim",
        "path": INTERIM_DIR / "oil_daily_clean.parquet",
        "expected_min_rows": 1,
    },
    {
        "file_name": "holidays_daily_clean.parquet",
        "expected_location": "data/interim",
        "path": INTERIM_DIR / "holidays_daily_clean.parquet",
        "expected_min_rows": 1,
    },
    {
        "file_name": "transactions_clean.parquet",
        "expected_location": "data/interim",
        "path": INTERIM_DIR / "transactions_clean.parquet",
        "expected_min_rows": 1,
    },
    {
        "file_name": "train_base.parquet",
        "expected_location": "data/processed",
        "path": PROCESSED_DIR / "train_base.parquet",
        "expected_min_rows": 1,
    },
    {
        "file_name": "test_base.parquet",
        "expected_location": "data/processed",
        "path": PROCESSED_DIR / "test_base.parquet",
        "expected_min_rows": 1,
    },
]


output_validation_rows = []

for output_file in expected_output_files:
    file_path = output_file["path"]
    file_exists = file_path.exists()

    if file_exists:
        df_preview = pd.read_parquet(file_path)
        row_count = len(df_preview)
        column_count = df_preview.shape[1]
    else:
        row_count = None
        column_count = None

    output_validation_rows.append(
        {
            "file_name": output_file["file_name"],
            "expected_location": output_file["expected_location"],
            "exists": file_exists,
            "rows": row_count,
            "columns": column_count,
            "file_size_mb": (
                round(file_path.stat().st_size / (1024**2), 2) if file_exists else None
            ),
            "path": str(file_path),
            "is_valid": (
                file_exists
                and row_count is not None
                and row_count >= output_file["expected_min_rows"]
            ),
        }
    )


final_outputs_validation_df = pd.DataFrame(output_validation_rows)

display(final_outputs_validation_df)

final_outputs_validation_df.to_csv(
    DATA_QUALITY_DIR / "final_outputs_validation.csv",
    index=False,
)


missing_or_invalid_outputs = final_outputs_validation_df.loc[
    ~final_outputs_validation_df["is_valid"],
    "file_name",
].tolist()

if missing_or_invalid_outputs:
    raise FileNotFoundError(
        "Missing or invalid generated output files: "
        + ", ".join(missing_or_invalid_outputs)
    )


,file_name,expected_location,exists,rows,columns,file_size_mb,path,is_valid
0,train_clean.parquet,data/interim,True,3000888,6,20.97,....,True
1,test_clean.parquet,data/interim,True,28512,5,0.18,....,True
2,stores_clean.parquet,data/interim,True,54,5,0.00,....,True
3,oil_daily_clean.parquet,data/interim,True,1704,5,0.02,....,True
4,holidays_daily_clean.parquet,data/interim,True,1704,17,0.03,....,True
5,transactions_clean.parquet,data/interim,True,83488,3,0.17,....,True
6,train_base.parquet,data/processed,True,3000888,30,21.42,....,True
7,test_base.parquet,data/processed,True,28512,29,0.20,....,True


# 28.Final Conclusions

## Data Engineering Summary

The raw datasets were successfully audited, validated, cleaned, and exported into reusable intermediate and processed files.

The original raw files were not modified. All generated outputs were saved under `data/interim/`, `data/processed/`, and `reports/data_quality/`.

The expected forecasting grain was validated as:


date + store_nbr + family

The main data contracts were successfully validated:

train and test preserve unique forecasting keys.
stores contains one record per store.
oil was converted into a complete daily table.
holidays was aggregated by date to avoid row multiplication.
transactions was cleaned and saved separately as a historical auxiliary table.
sample_submission matches the test identifiers.

The processed base tables were generated successfully:
data/processed/train_base.parquet
data/processed/test_base.parquet

- primary keys;
- train/test consistency;
- store references;
- missing values in critical columns;
- negative sales;
- negative promotions;
- negative transactions;
- invalid oil prices.
- Analytical Implications

The generated base tables are suitable for the next phase of the project: exploratory data analysis.

The most relevant variables for the forecasting problem are expected to include:

- historical sales;
- store metadata;
- product family;
- promotions;
- oil price;
- holiday and event indicators;
- calendar effects.

The transactions table should be analyzed carefully. It may be useful for historical business understanding, but it should not be used directly as a future-known feature unless a realistic forecasting strategy is defined later.

Business Relevance

The cleaned datasets support the business objective of forecasting store-level product family sales for a 16-day replenishment horizon.

Reliable data preparation is critical for this use case because data leakage, duplicated joins, or inconsistent keys could lead to distorted demand estimates and poor stock decisions.

Notebook Closure Criteria

This notebook is considered complete because:

- all expected raw files were found and loaded;
- schemas were validated;
- date ranges were audited;
- keys and duplicates were checked;
- missing values were audited;
- numeric and categorical variables were validated;
- train/test consistency was confirmed;
- referential integrity was reviewed;
- cleaning decisions were documented;
- clean intermediate datasets were generated;
- processed base tables were generated;
- data quality reports were exported;
- final outputs were validated.
